# EcoData | Organização e Qualidade de Dados Ambientais

**Python, Automação e Pandas.**

## 1. Bibliotecas utilizadas

Esta etapa importa as ferramentas necessárias ao processamento:

- **pathlib:** manipulação de caminhos, arquivos e diretórios.
- **pandas:** leitura, conversão e análise de dados em CSV e Excel.
- **hashlib:** cálculo de assinaturas SHA-256 para identificar arquivos idênticos.
- **shutil:** movimentação de arquivos para a pasta de arquivamento.

**Nenhum arquivo é modificado nesta etapa.**

In [84]:
from pathlib import Path
import pandas as pd
import hashlib
import shutil

## 2. Configuração dos diretórios

Esta etapa identifica a pasta principal do projeto e define os caminhos utilizados nas próximas operações.

**Diretórios principais:**

- `dados_recebidos`: armazena os arquivos originais, que devem permanecer intactos.
- `EcoData_Organizado`: recebe os arquivos classificados e padronizados em CSV.
- `EcoData_Duplicados_Arquivados`: guarda cópias redundantes durante o tratamento de duplicatas.

O programa utiliza caminhos relativos ao projeto para evitar dependência de um diretório absoluto.

**Resultado esperado:** apresentar os caminhos encontrados e confirmar a localização da pasta de origem.

A configuração dos caminhos não modifica os arquivos.

In [85]:
pasta_execucao = Path.cwd().resolve()

if (pasta_execucao / "dados_recebidos").is_dir():
    pasta_projeto = pasta_execucao
else:
    projetos_candidatos = [
        item for item in pasta_execucao.iterdir()
        if item.is_dir() and (item / "dados_recebidos").is_dir()
    ]

    if len(projetos_candidatos) != 1:
        raise FileNotFoundError(
            "Não foi possível identificar uma única pasta do EcoData a partir de "
            f"{pasta_execucao}. Execute o notebook na pasta do projeto ou em sua pasta pai."
        )

    pasta_projeto = projetos_candidatos[0].resolve()

pasta_origem = pasta_projeto / "dados_recebidos"
pasta_destino = pasta_projeto / "EcoData_Organizado"

if not pasta_origem.is_dir():
    raise FileNotFoundError(f"Pasta de origem não encontrada: {pasta_origem}")

print(f"Pasta do projeto: {pasta_projeto}")
print(f"Origem: {pasta_origem}")
print(f"Destino: {pasta_destino}")


Pasta do projeto: C:\Users\giova\Documents\Python\Projetos - Pandas, ML e Programas\6 - Automação e Organização de Dados Ambientais
Origem: C:\Users\giova\Documents\Python\Projetos - Pandas, ML e Programas\6 - Automação e Organização de Dados Ambientais\dados_recebidos
Destino: C:\Users\giova\Documents\Python\Projetos - Pandas, ML e Programas\6 - Automação e Organização de Dados Ambientais\EcoData_Organizado


## 3. Leitura e classificação dos arquivos

Nesta etapa, o programa define duas funções auxiliares:

**`listar_arquivos()`**

Percorre a pasta de origem e identifica arquivos nos formatos CSV e XLSX.

**`identificar_categoria()`**

Analisa as palavras presentes nos nomes dos arquivos e determina sua categoria:

| Categoria | Identificadores |
|---|---|
| Chuvas | chuva, precipitacao, pluviometria |
| Temperatura | temperatura, temp, termometria |
| Qualidade do Ar | ar, qualidade, poluicao, pm25 |
| Pendências | Arquivos não reconhecidos |

A classificação utiliza os nomes dos arquivos, não suas colunas internas.

**Importante:** essas funções apenas identificam informações. Não movem nem alteram os documentos.

In [86]:
def listar_arquivos(pasta_origem):
    pasta_origem = Path(pasta_origem)

    if not pasta_origem.is_dir():
        raise FileNotFoundError(
            f"Pasta de origem não encontrada: {pasta_origem}"
        )

    arquivos = []

    for arquivo in pasta_origem.iterdir():
        if arquivo.is_file() and arquivo.suffix.lower() in (".csv", ".xlsx"):
            arquivos.append(arquivo)

    return sorted(arquivos)

def identificar_categoria(arquivo):
    nome = arquivo.stem.lower().replace("-", "_")
    palavras = set(nome.split("_"))

    if palavras.intersection({"chuva", "precipitacao", "pluviometria"}):
        return "Chuvas"

    elif palavras.intersection({"temperatura", "temp", "termometria"}):
        return "Temperatura"

    elif palavras.intersection({"ar", "qualidade", "poluicao", "pm25"}):
        return "Qualidade_do_Ar"

    else:
        return "Pendencias"


### 4. Cálculo de hash, conversão e organização dos arquivos

Nesta etapa, o programa realiza o cálculo de hash dos arquivos, converte os documentos recebidos para o formato CSV e organiza os resultados nas pastas correspondentes às categorias ambientais.

A função `calcular_hash()` utiliza o algoritmo SHA-256 para gerar uma assinatura digital baseada no conteúdo de cada arquivo. Essa assinatura permite comparar documentos e identificar arquivos com conteúdo binário idêntico.

A função `converter_para_csv()` utiliza a biblioteca Pandas para ler arquivos CSV e Excel e salvar os dados em formato CSV, utilizando a codificação UTF-8-SIG e sem incluir o índice do DataFrame.

A função `organizar_arquivos()` identifica a categoria de cada documento e direciona os arquivos para as pastas `Chuvas`, `Temperatura`, `Qualidade_do_Ar` e `Pendencias`.

Durante a organização, o programa consulta os hashes dos arquivos ativos e dos arquivos anteriormente arquivados. Quando identifica que um conteúdo arquivado já possui uma cópia ativa, evita sua reimportação.

A função compara o CSV convertido com os destinos existentes. Conteúdos iguais são ignorados; conteúdos divergentes são preservados com nomes alternativos, sem sobrescrita. A publicação usa um arquivo temporário, e o resumo informa também os conflitos.

**Objetivo:** padronizar e organizar os arquivos ambientais, reduzir reprocessamentos desnecessários e evitar a recriação de duplicatas anteriormente arquivadas que ainda possuem uma cópia ativa.


In [87]:
# ==================================================
# CÁLCULO DE HASH SHA-256
# ==================================================

def calcular_hash(arquivo):
    hash_arquivo = hashlib.sha256()

    with open(arquivo, "rb") as arquivo_aberto:
        for bloco in iter(
            lambda: arquivo_aberto.read(8192),
            b""
        ):
            hash_arquivo.update(bloco)

    return hash_arquivo.hexdigest()


# ==================================================
# CONVERSÃO DE ARQUIVOS PARA CSV
# ==================================================

def converter_para_csv(arquivo, destino):

    if arquivo.suffix.lower() == ".csv":
        dados = pd.read_csv(
            arquivo,
            encoding="utf-8-sig"
        )

    else:
        dados = pd.read_excel(arquivo)

    dados.to_csv(
        destino,
        index=False,
        encoding="utf-8-sig"
    )


# ==================================================
# ORGANIZAÇÃO DOS ARQUIVOS
# ==================================================

def organizar_arquivos(pasta_origem, pasta_destino):
    import csv
    import io
    import os
    import tempfile

    pasta_destino = Path(pasta_destino)
    pasta_backup = pasta_destino.parent / "EcoData_Duplicados_Arquivados"
    categorias = ["Chuvas", "Temperatura", "Qualidade_do_Ar", "Pendencias"]
    contagem = {categoria: 0 for categoria in categorias}
    ignorados = 0
    conflitos = 0
    erros = 0

    for categoria in categorias:
        (pasta_destino / categoria).mkdir(parents=True, exist_ok=True)

    # Representa os campos CSV de forma estável entre LF e CRLF.
    # Mantém diferenças nos textos, na ordem das linhas e nos identificadores.
    def assinatura_csv(arquivo):
        buffer = io.StringIO(newline="")
        with Path(arquivo).open("r", encoding="utf-8-sig", newline="") as entrada:
            leitor = csv.reader(entrada, strict=True)
            cabecalho = next(leitor, None)
            if not cabecalho:
                raise ValueError("CSV vazio ou sem cabeçalho")
            escritor = csv.writer(buffer, lineterminator="\n")
            escritor.writerow(cabecalho)
            for linha in leitor:
                escritor.writerow(linha)
        return hashlib.sha256(buffer.getvalue().encode("utf-8")).hexdigest()

    arquivos_invalidos = []

    def hashes_csv_validos(pasta):
        hashes = set()
        if pasta.is_dir():
            for arquivo_csv in pasta.rglob("*.csv"):
                try:
                    hashes.add(assinatura_csv(arquivo_csv))
                except Exception as erro:
                    arquivos_invalidos.append((arquivo_csv, erro))
        return hashes

    hashes_arquivados = hashes_csv_validos(pasta_backup)
    hashes_ativos = hashes_csv_validos(pasta_destino)
    erros += len(arquivos_invalidos)
    for arquivo_invalido, erro in arquivos_invalidos:
        print(f"ERRO ao comparar {arquivo_invalido}: {erro}")

    for arquivo in listar_arquivos(pasta_origem):
        categoria = identificar_categoria(arquivo)
        destino = pasta_destino / categoria / f"{arquivo.stem}.csv"
        temporario = None

        try:
            # O arquivo temporário fica no mesmo diretório do destino.
            # No Windows, ele precisa estar fechado antes da conversão.
            with tempfile.NamedTemporaryFile(
                prefix=".ecodata_", suffix=".tmp",
                dir=destino.parent, delete=False
            ) as arquivo_temporario:
                temporario = Path(arquivo_temporario.name)

            converter_para_csv(arquivo, temporario)
            assinatura = assinatura_csv(temporario)

            conflito = False
            motivo = "conteúdo divergente ou incompleto"
            if os.path.lexists(destino):
                try:
                    assinatura_destino = assinatura_csv(destino)
                except Exception as erro_destino:
                    conflito = True
                    motivo = f"destino inválido ou ilegível ({erro_destino})"
                else:
                    if assinatura_destino == assinatura:
                        hashes_ativos.add(assinatura)
                        ignorados += 1
                        print(f"IGNORADO (já existe e foi validado): {destino.name}")
                        continue
                    conflito = True

            # Reconhece uma saída alternativa criada em execução anterior.
            alternativos = sorted(set(
                list(destino.parent.glob(f"{destino.stem}_origem_*.csv")) +
                list(destino.parent.glob(f"{destino.stem}_recuperado*.csv"))
            ))
            ja_preservado = None
            for alternativo in alternativos:
                try:
                    if assinatura_csv(alternativo) == assinatura:
                        ja_preservado = alternativo
                        break
                except Exception:
                    continue

            if ja_preservado is not None:
                if conflito:
                    conflitos += 1
                    print(
                        f"CONFLITO: origem {arquivo.name}; destino {destino.name} "
                        f"({motivo}); conteúdo já preservado em "
                        f"{ja_preservado.name}. Verifique o destino conflitante."
                    )
                else:
                    print(
                        f"IGNORADO (já presente em nome alternativo): "
                        f"{arquivo.name} -> {ja_preservado.name}"
                    )
                hashes_ativos.add(assinatura)
                ignorados += 1
                continue

            # Preserva EC-02: um conteúdo arquivado só é ignorado
            # quando outra cópia válida já está ativa.
            if assinatura in hashes_arquivados and assinatura in hashes_ativos:
                if conflito:
                    conflitos += 1
                    print(
                        f"CONFLITO: origem {arquivo.name}; destino {destino.name} "
                        f"({motivo}). Verifique o destino conflitante."
                    )
                ignorados += 1
                print(f"IGNORADO (conteúdo já presente na base ativa): {arquivo.name}")
                continue

            if conflito:
                if assinatura in hashes_arquivados:
                    nome_alternativo = f"{destino.stem}_recuperado"
                else:
                    nome_alternativo = (
                        f"{destino.stem}_origem_{arquivo.suffix.lower().lstrip('.')}"
                    )
                candidato = destino.with_name(f"{nome_alternativo}.csv")
            else:
                nome_alternativo = None
                candidato = destino
            numero = 2

            while True:
                if os.path.lexists(candidato):
                    try:
                        assinatura_candidato = assinatura_csv(candidato)
                    except Exception:
                        assinatura_candidato = None
                    if assinatura_candidato == assinatura:
                        hashes_ativos.add(assinatura)
                        ignorados += 1
                        if candidato != destino:
                            conflitos += 1
                            print(
                                f"CONFLITO: origem {arquivo.name}; destino "
                                f"{destino.name} ({motivo}); conteúdo já preservado "
                                f"em {candidato.name}. Verifique o destino conflitante."
                            )
                        else:
                            print(f"IGNORADO (já existe e foi validado): {destino.name}")
                        break
                    if candidato == destino:
                        conflito = True
                        motivo = "conteúdo divergente ou incompleto"
                        nome_alternativo = (
                            f"{destino.stem}_origem_{arquivo.suffix.lower().lstrip('.')}"
                        )
                        candidato = destino.with_name(f"{nome_alternativo}.csv")
                    else:
                        candidato = destino.with_name(
                            f"{nome_alternativo}_{numero}.csv"
                        )
                        numero += 1
                    continue

                try:
                    # Um hard link no mesmo volume publica sem substituir
                    # um destino criado por outra execução.
                    os.link(temporario, candidato)
                except FileExistsError:
                    continue

                hashes_ativos.add(assinatura)
                contagem[categoria] += 1
                if conflito:
                    conflitos += 1
                    print(
                        f"CONFLITO: origem {arquivo.name}; destino {destino.name} "
                        f"({motivo}); nova saída {categoria}/{candidato.name}. "
                        "Verifique o destino conflitante."
                    )
                else:
                    print(f"{arquivo.name}  ->  {categoria}/{candidato.name}")
                break

        except Exception as erro:
            erros += 1
            print(f"ERRO ao processar {arquivo.name}: {erro}")
        finally:
            if temporario is not None and temporario.exists():
                try:
                    temporario.unlink()
                except OSError as erro_limpeza:
                    erros += 1
                    print(f"ERRO ao remover temporário {temporario.name}: {erro_limpeza}")

    print("\nRESUMO DA ORGANIZAÇÃO")
    for categoria, total in contagem.items():
        print(f"{categoria}: {total}")
    print(f"Ignorados (existentes ou redundantes): {ignorados}")
    print(f"Conflitos: {conflitos}")
    print(f"Erros: {erros}")


# ==================================================
# EXECUÇÃO
# ==================================================

organizar_arquivos(
    pasta_origem,
    pasta_destino
)


IGNORADO (já existe): ar_copacabana_2026-10-18.csv
IGNORADO (já existe): arquivo_sem_identificacao.csv
IGNORADO (já existe): chuva_bangu_2026-10-16.csv
IGNORADO (já existe): chuva_meier_2026-10-07.csv
IGNORADO (conteúdo já presente na base ativa): chuva_tijuca_2026-10-03.csv
IGNORADO (já existe): coleta_turno_b.csv
IGNORADO (já existe): dados_estacao_04.csv
IGNORADO (já existe): pluviometria_copacabana_2026-10-14.csv
IGNORADO (já existe): pm25_tijuca_2026-10-11.csv
IGNORADO (conteúdo já presente na base ativa): poluicao_centro_2026-10-05_repasse.csv
IGNORADO (já existe): poluicao_madureira_2026-10-20.csv
IGNORADO (já existe): precipitacao_centro_2026-10-10.csv
IGNORADO (já existe): precipitacao_madureira_2026-10-19.csv
IGNORADO (conteúdo já presente na base ativa): precipitacao_tijuca_2026-10-03_repasse.csv
IGNORADO (conteúdo já presente na base ativa): qualidade_ar_centro_2026-10-05.csv
IGNORADO (já existe): qualidade_ar_meier_2026-10-15.csv
IGNORADO (já existe): relatorio_2026-10-14.

### 5. Identificação de arquivos duplicados

Nesta etapa, o programa verifica a existência de arquivos duplicados dentro da pasta de dados ambientais organizados.

A função `verificar_arquivos_duplicados()` percorre os arquivos CSV das diferentes categorias e utiliza a função `calcular_hash()`, definida na etapa anterior, para comparar suas assinaturas SHA-256.

Quando dois arquivos apresentam o mesmo hash, o programa identifica o conteúdo duplicado e informa os nomes dos documentos envolvidos, mesmo que possuam nomes diferentes.

Ao final da verificação, é apresentado um relatório com a quantidade total de arquivos analisados, o número de arquivos duplicados e a quantidade de conteúdos únicos encontrados.

Esta etapa realiza somente a identificação das duplicatas, sem excluir ou movimentar arquivos. O tratamento e o arquivamento das cópias redundantes são realizados em uma etapa específica do programa.

**Objetivo:** identificar conteúdos repetidos na base organizada, evitando que informações redundantes sejam consideradas registros independentes em análises posteriores.


In [88]:
# ==================================================
# VERIFICAÇÃO DE ARQUIVOS DUPLICADOS
# ==================================================

def verificar_arquivos_duplicados(pasta_destino):

    print("\nVERIFICAÇÃO DE ARQUIVOS DUPLICADOS")
    print("=" * 50)

    pasta = Path(pasta_destino)

    arquivos = sorted(
        pasta.rglob("*.csv")
    )

    hashes = {}
    duplicados = []

    for arquivo in arquivos:

        assinatura = calcular_hash(arquivo)

        if assinatura in hashes:

            arquivo_original = hashes[assinatura]

            duplicados.append(
                (arquivo_original, arquivo)
            )

            print("\nDUPLICATA ENCONTRADA")

            print(
                f"Arquivo original: "
                f"{arquivo_original.name}"
            )

            print(
                f"Arquivo duplicado: "
                f"{arquivo.name}"
            )

        else:

            hashes[assinatura] = arquivo

    print("\nRESUMO DA VERIFICAÇÃO")
    print("-" * 50)

    print(
        f"Arquivos analisados: {len(arquivos)}"
    )

    print(
        f"Arquivos duplicados: {len(duplicados)}"
    )

    print(
        f"Arquivos únicos: {len(hashes)}"
    )

    if not duplicados:
        print(
            "\nNenhum arquivo duplicado encontrado."
        )

    return duplicados


# ==================================================
# EXECUÇÃO
# ==================================================

arquivos_duplicados = verificar_arquivos_duplicados(
    pasta_destino
)



VERIFICAÇÃO DE ARQUIVOS DUPLICADOS

RESUMO DA VERIFICAÇÃO
--------------------------------------------------
Arquivos analisados: 22
Arquivos duplicados: 0
Arquivos únicos: 22

Nenhum arquivo duplicado encontrado.


## 6. Tratamento de arquivos duplicados

Após identificar documentos com conteúdo idêntico, esta etapa permite retirar cópias redundantes da base de análise.

A função `remover_arquivos_duplicados()` possui dois modos:

**Simulação — `executar=False`**

Identifica as duplicatas e informa o tratamento previsto, sem modificar os arquivos.

**Execução — `executar=True`**

Retira as cópias redundantes de `EcoData_Organizado`.

Quando ainda não existe uma cópia idêntica no backup, o arquivo é movido para `EcoData_Duplicados_Arquivados`.

Quando já existe um backup com o mesmo SHA-256, a cópia redundante da pasta organizada é excluída após a confirmação do conteúdo arquivado, sem criar outro backup idêntico.

O arquivo original em `dados_recebidos` é preservado.

**Verificação de sucesso**

Após uma remoção efetiva, o programa pode executar novamente a verificação SHA-256.

Na situação esperada, a pasta organizada permanece com 22 arquivos e nenhuma duplicata exata.

**Atenção:** mantenha `executar=False` como configuração padrão para evitar remoções involuntárias ao utilizar a opção "Executar Todas as Células".


In [89]:
def remover_arquivos_duplicados(pasta_destino, executar=False):

    pasta_destino = Path(pasta_destino)

    if not pasta_destino.is_dir():
        raise FileNotFoundError(
            f"Pasta não encontrada: {pasta_destino}"
        )

    pasta_backup = (
        pasta_destino.parent / "EcoData_Duplicados_Arquivados"
    )

    arquivos = sorted(pasta_destino.rglob("*.csv"))

    grupos = {}
    duplicados = []
    backups_por_hash = {}

    # Verifica os arquivos que já estão arquivados
    if pasta_backup.is_dir():

        for arquivo_backup in sorted(pasta_backup.rglob("*.csv")):

            assinatura = calcular_hash(arquivo_backup)

            backups_por_hash.setdefault(
                assinatura,
                arquivo_backup
            )

    # Agrupa os arquivos organizados pelo conteúdo
    for arquivo in arquivos:

        assinatura = calcular_hash(arquivo)

        grupos.setdefault(
            assinatura,
            []
        ).append(arquivo)

    print("\nREMOÇÃO DE ARQUIVOS DUPLICADOS")
    print("=" * 50)

    for assinatura, grupo in grupos.items():

        if len(grupo) < 2:
            continue

        # Prioriza manter arquivos sem "repasse"
        grupo.sort(
            key=lambda arquivo: (
                "repasse" in arquivo.stem.lower(),
                str(arquivo).lower()
            )
        )

        arquivo_mantido = grupo[0]

        for arquivo_duplicado in grupo[1:]:

            duplicados.append(arquivo_duplicado)

            print(f"\nArquivo mantido: {arquivo_mantido.name}")
            print(f"Duplicata: {arquivo_duplicado.name}")

            # Verifica se o conteúdo já existe no backup
            if assinatura in backups_por_hash:

                arquivo_no_backup = backups_por_hash[assinatura]

                if executar:

                    # Confirma a integridade do backup
                    if (
                        not arquivo_no_backup.is_file()
                        or calcular_hash(arquivo_no_backup) != assinatura
                    ):
                        raise RuntimeError(
                            f"Backup não pôde ser confirmado: {arquivo_no_backup}"
                        )

                    # Remove somente a cópia redundante
                    # da pasta EcoData_Organizado
                    arquivo_duplicado.unlink()

                    print(
                        "Status: backup idêntico já existe; "
                        "cópia redundante removida da pasta organizada"
                    )

                else:

                    print(
                        "Status: backup idêntico já existe; "
                        "nenhuma alteração (simulação)"
                    )

                continue

            # Simulação sem modificar arquivos
            if not executar:

                print(
                    "Status: identificado, sem alterações"
                )

                continue

            # Arquiva apenas quando ainda não existe
            # uma cópia idêntica no backup
            destino_backup = (
                pasta_backup /
                arquivo_duplicado.relative_to(pasta_destino)
            )

            destino_backup.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            destino_final = destino_backup
            contador = 2

            # Evita sobrescrever nomes já existentes
            while destino_final.exists():

                destino_final = destino_backup.with_name(
                    f"{destino_backup.stem}_{contador}"
                    f"{destino_backup.suffix}"
                )

                contador += 1

            shutil.move(
                str(arquivo_duplicado),
                str(destino_final)
            )

            # Atualiza o controle dos backups
            backups_por_hash[assinatura] = destino_final

            print("Status: movido para backup")

    print("\nRESUMO")
    print("-" * 50)

    print(f"Arquivos analisados: {len(arquivos)}")
    print(f"Duplicatas encontradas: {len(duplicados)}")

    if executar:

        restantes = len(
            list(pasta_destino.rglob("*.csv"))
        )

        print(f"Arquivos restantes: {restantes}")

    else:

        print(
            "Modo de simulação: "
            "nenhum arquivo foi movido ou removido"
        )

    return duplicados


# EXECUÇÃO DA FUNÇÃO

duplicados = remover_arquivos_duplicados(
    pasta_destino,
    executar=False # Selecionar True para remover arquivos duplicados ou False para simular a remoção 
)

duplicados_restantes = verificar_arquivos_duplicados(
    pasta_destino
)

# Apresenta a mensagem conforme o resultado
if len(duplicados_restantes) == 0:

    print("\nSUCESSO!")
    print("A base EcoData está livre de arquivos duplicados exatos.")

else:

    print("\nATENÇÃO!")
    print(f"Foram encontrados {len(duplicados_restantes)} arquivos duplicados.")


REMOÇÃO DE ARQUIVOS DUPLICADOS

RESUMO
--------------------------------------------------
Arquivos analisados: 22
Duplicatas encontradas: 0
Modo de simulação: nenhum arquivo foi movido ou removido

VERIFICAÇÃO DE ARQUIVOS DUPLICADOS

RESUMO DA VERIFICAÇÃO
--------------------------------------------------
Arquivos analisados: 22
Arquivos duplicados: 0
Arquivos únicos: 22

Nenhum arquivo duplicado encontrado.

SUCESSO!
A base EcoData está livre de arquivos duplicados exatos.


## 7. Inspeção inicial da qualidade dos dados

Esta etapa utiliza Pandas para examinar a estrutura de cada arquivo CSV organizado.

A função `verificar_qualidade()` apresenta:

- Nome do arquivo e categoria.
- Quantidade de registros.
- Nomes das colunas.
- Quantidade de valores ausentes.
- Quantidade de linhas duplicadas internamente.

**Limitação importante**

O método `dados.duplicated()` identifica registros repetidos dentro de um mesmo DataFrame.

Ele não compara diretamente os registros de diferentes arquivos.

A verificação não altera nenhuma informação e serve como diagnóstico inicial da qualidade das bases.


In [90]:
def verificar_qualidade(pasta_destino):
    pasta_destino = Path(pasta_destino)

    print("\nRELATÓRIO INICIAL DE QUALIDADE")
    print("=" * 50)

    categorias = [
        "Chuvas",
        "Temperatura",
        "Qualidade_do_Ar",
        "Pendencias"
    ]

    for categoria in categorias:
        pasta = pasta_destino / categoria

        for arquivo in sorted(pasta.glob("*.csv")):
            try:
                dados = pd.read_csv(arquivo)

                print(f"\nArquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Linhas: {len(dados)}")
                print(f"Colunas: {list(dados.columns)}")

                ausentes = dados.isna().sum().sum()
                duplicados = dados.duplicated().sum()

                print(f"Valores ausentes: {ausentes}")
                print(f"Linhas duplicadas: {duplicados}")

            except Exception as erro:
                print(f"Erro ao ler {arquivo.name}: {erro}")

verificar_qualidade(pasta_destino)



RELATÓRIO INICIAL DE QUALIDADE

Arquivo: chuva_bangu_2026-10-16.csv
Categoria: Chuvas
Linhas: 40
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: chuva_meier_2026-10-07.csv
Categoria: Chuvas
Linhas: 38
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 2
Linhas duplicadas: 0

Arquivo: chuva_tijuca_2026-10-03.csv
Categoria: Chuvas
Linhas: 32
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: pluviometria_copacabana_2026-10-14.csv
Categoria: Chuvas
Linhas: 30
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: precipitacao_centro_2026-10-10.csv
Categoria: Chuvas
Linhas: 28
Colunas: ['id_registro', 'data_hora', 'estacao_id', '

## 8. Investigação dos valores ausentes

Esta etapa identifica quais colunas e registros possuem informações faltantes, representadas pelo Pandas como `NaN`.

A função `detalhar_ausentes()` mostra o nome do arquivo, as colunas afetadas e as linhas que precisam ser investigadas.

**Resultados identificados na análise inicial:**

| Arquivo | Coluna afetada | Ausentes |
|---|---|---:|
| chuva_meier_2026-10-07.csv | precipitacao_mm | 2 |
| temp_bangu_2026-10-20.csv | umidade_relativa_pct | 1 |
| pm25_tijuca_2026-10-11.csv | pm25_ug_m3 | 1 |

**Total histórico: 4 valores ausentes.**

Um valor ausente não deve ser interpretado automaticamente como zero.

O tratamento depende do significado da variável e do objetivo da análise.

Esta etapa apenas identifica os problemas, preservando os dados originais.


In [91]:
def detalhar_ausentes(pasta_destino):
    pasta_destino = Path(pasta_destino)

    categorias = [
        "Chuvas",
        "Temperatura",
        "Qualidade_do_Ar",
        "Pendencias"
    ]

    print("\nINVESTIGAÇÃO DE VALORES AUSENTES")
    print("=" * 50)
    lidos = 0
    falhas = []

    for categoria in categorias:
        pasta = pasta_destino / categoria

        for arquivo in sorted(pasta.glob("*.csv")):
            try:
                dados = pd.read_csv(arquivo)
            except UnicodeDecodeError as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE DECODIFICAÇÃO]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: confira o formato do original e descubra sua codificação.")
                print("Crie uma cópia em UTF-8-SIG sem alterar o original; confira estrutura e registros.")
                print("Reprocesse a cópia pelo fluxo seguro do EcoData e confira a categoria.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue
            except (pd.errors.ParserError, pd.errors.EmptyDataError, OSError) as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE LEITURA]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Tipo: {type(erro).__name__}")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: verifique a estrutura, o conteúdo e o acesso ao CSV.")
                print("Corrija uma cópia, valide sua leitura e reprocesse pelo fluxo seguro do EcoData.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue

            lidos += 1

            ausentes = dados.isna().sum()

            if ausentes.sum() > 0:
                print(f"\nArquivo: {arquivo.name}")

                print("\nColunas com valores ausentes:")
                print(ausentes[ausentes > 0])

                print("\nRegistros afetados:")
                registros = dados[
                    dados.isna().any(axis=1)
                ]

                print(registros.to_string(index=False))

    print("\nRESUMO DA LEITURA")
    print(f"Arquivos lidos com sucesso: {lidos}")
    print(f"Arquivos com erro de leitura: {len(falhas)}")
    if falhas:
        print("Arquivos não analisados:")
        for categoria_falha, caminho_falha in falhas:
            print(f"- {categoria_falha}: {caminho_falha}")

detalhar_ausentes(pasta_destino)



INVESTIGAÇÃO DE VALORES AUSENTES

Arquivo: chuva_meier_2026-10-07.csv

Colunas com valores ausentes:
precipitacao_mm    2
dtype: int64

Registros afetados:
        id_registro        data_hora estacao_id bairro  precipitacao_mm periodo_medicao
PLU-ME-20261007-009 2026-10-07 04:00  PLU-ME-01  Méier              NaN          30 min
PLU-ME-20261007-022 2026-10-07 10:30  PLU-ME-01  Méier              NaN          30 min

Arquivo: temp_bangu_2026-10-20.csv

Colunas com valores ausentes:
umidade_relativa_pct    1
dtype: int64

Registros afetados:
        id_registro        data_hora estacao_id bairro  temperatura_c  umidade_relativa_pct
TMP-BG-20261020-013 2026-10-20 06:00  TMP-BG-01  Bangu           32.5                   NaN

Arquivo: pm25_tijuca_2026-10-11.csv

Colunas com valores ausentes:
pm25_ug_m3    1
dtype: int64

Registros afetados:
       id_registro        data_hora estacao_id bairro  pm25_ug_m3  pm10_ug_m3
AR-TJ-20261011-007 2026-10-11 03:00   AR-TJ-01 Tijuca         NaN       

## 9. Validação das informações ambientais

Além de identificar valores ausentes, precisamos verificar se os dados respeitam as regras de validade definidas para cada variável.

A função `validar_valores()` utiliza os seguintes critérios:

| Variável | Regra |
|---|---|
| Precipitação (mm) | Não pode ser negativa |
| Umidade relativa (%) | Entre 0 e 100 |
| PM2,5 (µg/m³) | Não pode ser negativo |
| PM10 (µg/m³) | Não pode ser negativo |

A função também identifica valores não numéricos nas colunas verificadas e informa quando uma coluna obrigatória para a regra está ausente.

**Interpretação do resultado**

Se somente o cabeçalho `VALIDAÇÃO DE VALORES` aparecer, nenhuma violação das regras implementadas foi encontrada.

Isso não significa que o conjunto de dados esteja completamente livre de inconsistências.

Os registros não são corrigidos ou excluídos automaticamente.


In [92]:
def validar_valores(pasta_destino):
    pasta_destino = Path(pasta_destino)

    regras = {
        "Chuvas": {
            "precipitacao_mm": (0, None)
        },
        "Temperatura": {
            "umidade_relativa_pct": (0, 100)
        },
        "Qualidade_do_Ar": {
            "pm25_ug_m3": (0, None),
            "pm10_ug_m3": (0, None)
        }
    }

    print("\nVALIDAÇÃO DE VALORES")
    print("=" * 50)
    lidos = 0
    falhas = []

    for categoria, colunas in regras.items():
        pasta = pasta_destino / categoria

        for arquivo in sorted(pasta.glob("*.csv")):
            try:
                dados = pd.read_csv(arquivo)
            except UnicodeDecodeError as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE DECODIFICAÇÃO]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: confira o formato do original e descubra sua codificação.")
                print("Crie uma cópia em UTF-8-SIG sem alterar o original; confira estrutura e registros.")
                print("Reprocesse a cópia pelo fluxo seguro do EcoData e confira a categoria.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue
            except (pd.errors.ParserError, pd.errors.EmptyDataError, OSError) as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE LEITURA]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Tipo: {type(erro).__name__}")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: verifique a estrutura, o conteúdo e o acesso ao CSV.")
                print("Corrija uma cópia, valide sua leitura e reprocesse pelo fluxo seguro do EcoData.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue

            lidos += 1

            for coluna, limites in colunas.items():
                if coluna not in dados.columns:
                    print(f"{arquivo.name}: coluna {coluna} ausente")
                    continue

                minimo, maximo = limites

                valores = pd.to_numeric(
                    dados[coluna],
                    errors="coerce"
                )

                nao_numericos = (
                    dados[coluna].notna() &
                    valores.isna()
                )

                invalidos = valores < minimo

                if maximo is not None:
                    invalidos = invalidos | (valores > maximo)

                problemas = invalidos | nao_numericos

                if problemas.any():
                    print(f"\nArquivo: {arquivo.name}")
                    print(f"Coluna: {coluna}")

                    print(f"Valores problemáticos: {problemas.sum()}")

                    print(
                        dados.loc[
                            problemas,
                            ["id_registro", coluna]
                        ].to_string(index=False)
                    )

    print("\nRESUMO DA LEITURA")
    print(f"Arquivos lidos com sucesso: {lidos}")
    print(f"Arquivos com erro de leitura: {len(falhas)}")
    if falhas:
        print("Arquivos não analisados:")
        for categoria_falha, caminho_falha in falhas:
            print(f"- {categoria_falha}: {caminho_falha}")

validar_valores(pasta_destino)



VALIDAÇÃO DE VALORES


# EcoData | Organização e qualidade de dados ambientais

## Objetivo

Organizar os arquivos ambientais recebidos por categoria, converter planilhas Excel para CSV e inspecionar a qualidade dos dados sem alterar os arquivos de origem.

## Funcionalidades concluídas

- Listagem de arquivos CSV e XLSX e classificação por nome em **Chuvas**, **Temperatura**, **Qualidade_do_Ar** e **Pendencias**.
- Conversão de XLSX para CSV UTF-8 com BOM e organização por categoria. Os originais são preservados. Destinos iguais são reconhecidos; conteúdos divergentes recebem nomes alternativos sem sobrescrita.
- Identificação de arquivos binariamente idênticos com SHA-256 e relatório dos pares encontrados.
- Simulação de tratamento de duplicatas. O modo padrão não move arquivos; `executar=True` arquiva redundantes em `EcoData_Duplicados_Arquivados`, fora de `EcoData_Organizado`, priorizando nomes sem `repasse` e escolhendo um nome livre para não sobrescrever arquivos arquivados.
- Relatórios de qualidade, investigação de valores ausentes e validação dos limites ambientais configurados.

## Significado dos relatórios

A organização informa quantos arquivos foram convertidos, ignorados, conflitantes e com erro. A inspeção de qualidade mostra arquivo, categoria, linhas, colunas, valores ausentes e linhas repetidas dentro de cada arquivo. A investigação de ausentes lista as colunas e os registros afetados. A validação imprime ocorrências apenas para valores abaixo/acima dos limites, dados não numéricos em colunas verificadas ou colunas esperadas ausentes. Ausência do cabeçalho de violações significa somente que as regras verificadas não encontraram problemas.

A verificação SHA-256 compara o conteúdo binário integral dos CSVs, independentemente do nome. Antes de eventual arquivamento, os dados observados eram **24 arquivos: 2 duplicatas e 22 conteúdos únicos**. Os pares registrados foram `chuva_tijuca_2026-10-03.csv` / `precipitacao_tijuca_2026-10-03_repasse.csv` e `poluicao_centro_2026-10-05_repasse.csv` / `qualidade_ar_centro_2026-10-05.csv`.

## Resultados observados no estado inicial

| Categoria | Arquivos | Registros |
|:--|--:|--:|
| Chuvas | 7 | 228 |
| Temperatura | 7 | 234 |
| Qualidade_do_Ar | 6 | 184 |
| Pendencias | 4 | 108 |
| **Total** | **24** | **754** |

Foram observados **4 valores ausentes**: dois em `precipitacao_mm` de `chuva_meier_2026-10-07.csv`, um em `umidade_relativa_pct` de `temp_bangu_2026-10-20.csv` e um em `pm25_ug_m3` de `pm25_tijuca_2026-10-11.csv`. A validação anterior não encontrou violações para precipitação não negativa, umidade entre 0 e 100, `pm25_ug_m3` e `pm10_ug_m3` não negativos. Esses dados ausentes não foram preenchidos nem removidos.

No inventário histórico anterior ao arquivamento havia 24 CSVs organizados e 2 conteúdos duplicados. No inventário consultado para R1, `EcoData_Organizado` contém 22 CSVs e `EcoData_Duplicados_Arquivados` contém 2 cópias redundantes. Esses números descrevem estados observados em momentos distintos; cada execução deve informar as contagens reais das pastas.

## Cuidados com duplicatas

A célula de tratamento executa por padrão com `executar=False`, portanto apenas simula. A remoção efetiva é uma opção manual: ao usar `executar=True`, os arquivos redundantes são movidos para a pasta de arquivamento, sem exclusão definitiva. Após remoção efetiva, espera-se que a pasta organizada tenha 22 CSVs e a pasta de arquivamento tenha 2; uma nova verificação deverá encontrar zero duplicatas exatas restantes.

A pasta `dados_recebidos` mantém os 24 originais. A organização consulta os conteúdos ativos e arquivados para evitar reimportar uma cópia redundante quando já há outra cópia ativa. Se não houver cópia ativa, permite a recuperação. Quando o nome de destino já está ocupado por outro conteúdo, preserva ambos em caminhos diferentes. O modo padrão de execução completa nunca arquiva automaticamente.

## Limitações atuais e próximas etapas

A classificação depende dos identificadores presentes no nome do arquivo. A comparação de duplicatas cobre CSVs pelo conteúdo binário, e a validação cobre somente as colunas e limites explicitamente configurados. As próximas etapas podem incluir regras de classificação revisáveis, registro persistente de processamento e validações de esquema mais amplas, mantendo a decisão humana sobre tratamento de valores ausentes.
